# B2-024-gpu-scientific-ml-capstone — Practice p09 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** semi-supervised-pseudo-labeling

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import torch

PROBS = torch.tensor([[0.90, 0.05, 0.05], [0.20, 0.70, 0.10], [0.85, 0.10, 0.05], [0.05, 0.15, 0.80],
                      [0.90, 0.06, 0.04], [0.45, 0.45, 0.10], [0.10, 0.88, 0.02], [0.02, 0.08, 0.90],
                      [0.80, 0.10, 0.10], [0.05, 0.88, 0.07]], dtype=torch.float64)

## Solution

The rules of Session 4, §3, translate directly:

1. `probs.argmax(dim=1)` gives each row's label and `probs.max(dim=1).values` its confidence; `torch.argmax` returns the **first** maximal index, so arg-max ties go to the lower class index.
2. Candidates satisfy `conf >= threshold` (equality counts).
3. Within each class, a **stable** sort of the candidates by descending confidence keeps the ascending row order among equal confidences, so confidence ties go to the lower row index; the first `max_per_class` are kept.
4. The kept indices are sorted ascending and their labels gathered in the same order.

Nothing is written into `probs`, and the empty case falls out naturally as `int64` tensors of shape `(0,)`.

In [ ]:
def select_pseudo_labels(probs, threshold, max_per_class, *, strict=False, use_cap=True):
    pred = probs.argmax(dim=1)                        # argmax returns the first maximal index on ties
    conf = probs.max(dim=1).values
    candidate = conf > threshold if strict else conf >= threshold
    kept = []
    for k in range(probs.shape[1]):
        rows = torch.nonzero(candidate & (pred == k)).flatten()        # ascending row order
        order = torch.sort(-conf[rows], stable=True).indices            # high confidence first, ties by row
        rows = rows[order]
        kept.append(rows[:max_per_class] if use_cap else rows)
    indices = torch.sort(torch.cat(kept)).values.to(torch.int64)
    labels = pred[indices].to(torch.int64)
    return indices, labels


def wrong_strict(probs, threshold, max_per_class):
    return select_pseudo_labels(probs, threshold, max_per_class, strict=True)


def wrong_no_cap(probs, threshold, max_per_class):
    return select_pseudo_labels(probs, threshold, max_per_class, use_cap=False)


TABLE = [
    (0.80, 2, [0, 3, 4, 6, 7, 9], [0, 2, 0, 1, 2, 1]),
    (0.80, 1, [0, 6, 7], [0, 1, 2]),
    (0.80, 10, [0, 2, 3, 4, 6, 7, 8, 9], [0, 0, 2, 0, 1, 2, 0, 1]),
    (0.45, 10, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [0, 1, 0, 2, 0, 0, 1, 2, 0, 1]),
    (0.85, 2, [0, 4, 6, 7, 9], [0, 0, 1, 2, 1]),
    (0.95, 3, [], []),
]
PROBS_BEFORE = PROBS.clone()


def row_ok(fn, row):
    tau, cap, want_idx, want_lab = row
    idx, lab = fn(PROBS, tau, cap)
    return (idx.dtype == torch.int64 and lab.dtype == torch.int64
            and torch.equal(idx, torch.tensor(want_idx, dtype=torch.int64))
            and torch.equal(lab, torch.tensor(want_lab, dtype=torch.int64)))


results = [row_ok(select_pseudo_labels, row) for row in TABLE]
strict_results = [row_ok(wrong_strict, row) for row in TABLE]
nocap_results = [row_ok(wrong_no_cap, row) for row in TABLE]
empty_idx, empty_lab = select_pseudo_labels(PROBS, 0.95, 3)
for row in TABLE:
    print(row[:2], [t.tolist() for t in select_pseudo_labels(PROBS, row[0], row[1])])
print("correct:", results, "| '>' variant:", strict_results, "| no-cap variant:", nocap_results)

**Which rows each wrong variant fails.**
The `>` variant drops row 8 (confidence exactly $0.80$) at $\tau=0.80$, cap 10, and row 5 (confidence exactly $0.45$) at $\tau=0.45$.
The no-cap variant keeps all class-0 candidates in the cap-2 and cap-1 rows (e.g. rows 0, 2, 4, 8 at $\tau=0.80$, cap 2).

**Why rows 5 and 9 (row 9 paired with row 6) test the tie rules.**
Row 5 is $(0.45, 0.45, 0.10)$, an exact arg-max tie, so only the "lower class index" rule gives it label 0 (visible in the $\tau=0.45$ row of the table).
Row 9 has confidence $0.88$ for class 1, exactly equal to row 6, so with cap 1 only the "lower row index" rule decides that row 6 is kept and row 9 is dropped (the $\tau=0.80$, cap-1 row of the table).

### Answer check

In [ ]:
assert all(results)
assert empty_idx.shape == (0,) and empty_lab.shape == (0,)
assert empty_idx.dtype == torch.int64 and empty_lab.dtype == torch.int64
assert torch.equal(PROBS, PROBS_BEFORE)
assert not all(strict_results) and not all(nocap_results)
assert strict_results[2] is False and strict_results[3] is False
assert nocap_results[0] is False and nocap_results[1] is False